In [ ]:
!pip install mittens
!pip install scipy
from mittens import GloVe,Mittens
from scipy.sparse import *
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import csv

# Load GloVe embedding
def load_glove_embeddings(glove_file):
    embeddings_index = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    return embeddings_index

# Converting tweets to GloVe vectors
def tweet_to_glove_vector(tweet, embeddings_index, embedding_dim=100):
    words = tweet.split()
    valid_words = [word for word in words if word in embeddings_index]
    if not valid_words:
        return np.zeros(embedding_dim)
    return np.mean([embeddings_index[word] for word in valid_words], axis=0)


# Path to GloVe embedding (pretrained or our self trained embedding by running Training_GloVe code)
glove_file = '/content/drive/MyDrive/glove.twitter.27B.200d.txt'

embeddings_index = load_glove_embeddings(glove_file)


# Load preprocessed datasets from google drive
with open('/content/drive/MyDrive/train_pos_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_pos_content = file.readlines()

with open('/content/drive/MyDrive/train_neg_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_neg_content = file.readlines()

# Prepare dataframes
train_pos = pd.DataFrame(train_pos_content, columns=['tweet'])
train_pos['label'] = 1

train_neg = pd.DataFrame(train_neg_content, columns=['tweet'])
train_neg['label'] = 0

train = pd.concat([train_pos, train_neg], ignore_index = True)

tweets = train['tweet'].tolist()
embedding_dim = 200
tweet_vectors = np.array([tweet_to_glove_vector(tweet, embeddings_index, embedding_dim) for tweet in tweets])

y = train['label'].values

# 90% training set, 10% validation set
X_train, X_val, y_train, y_val = train_test_split(tweet_vectors, y, test_size=0.1, random_state=42)

# Logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))